# Frozen-model residual investigation

The seven-variable forest is unchanged. This notebook uses its **held-network predictions**, with fixed settings selected previously. This is descriptive follow-up on repeatedly examined data, not an independent validation or evidence that new predictors improve performance.

**37 / CB001_bubble_3 remains the dominant squared-error case:** centroid shift −447.69 min, predicted −82.89 min; 44.3% of equal-network weighted squared error. Excluding it from evaluation raises centroid R² **0.6145 → 0.7134**, MAE **2.822 → 2.761 min**. Peak-transfer R² changes **0.4910 → 0.5403**. No models are retrained for this exclusion. The earlier nested-selection score remains separate.

**The screening result is modest:** none of 92 unused geometric representations has a strong, consistent signed-residual association. Amount of channel length and complexity removed identify larger absolute errors, but largely do not identify the direction of correction or high percentage error. These are risk indicators until added-predictor experiments show otherwise.

**Shortlist:** width-SD change, parent route entropy, total-length change and cycle-rank change. Candidate selection is exploratory; any later gains need grouped validation and ideally selection inside training folds or new networks. A new predictor alone cannot remove a forest's training-response-range limit for case 37.

## Current frozen-model status and prediction coverage — 19 October 2026

The retained model is the **seven-input forest predicting full centroid shift**. Exposure and resistance are excluded. Entropy remains ln(1+Hc/Hp); neither ΔH nor parent entropy nor the three residual-screen additions was adopted. The frozen settings and artifact remain unchanged. All six follow-up experiments are complete; updated frozen-model SHAP, statistical inference and the companion regression/GAM are **planned, not completed**. Historical SHAP/permutation results refer to the earlier relative-sweep forests.

Prediction coverage verified by a one-to-one join against the frozen input: **2,170/2,170 matched**, no missing or extra prediction keys, across 15 examples.

| Collapse category | Included held-out predictions |
|---|---:|
| Simple | 258 |
| Compound member | 1,467 |
| Island | 268 |
| Full compound | 177 |
| Whole-network single centerline (`fully_collapsed`) | **0** |

**Full compound collapse and whole-network collapse are different.** All 15 networks have full compound comparisons in this cohort. The 15 whole-network single-centerline response records exist in `hydrograph_response/responses.parquet`, but are absent from the current predictor input and frozen predictions. Thus the frozen baseline scores do not evaluate those whole-network states. A subsequent separate held-network prediction check is now complete (see whole-network section below); these 15 cases remain outside baseline training and scores. The response table has 2,187 rows: the current predictor cohort omits those 15 plus `37/island_37_098` and `41/S061_unit_57`. This coverage audit establishes absence, not the reason for the two additional exclusions. Resolve their provenance before making claims about complete coverage. Audit files: [response_cohort_audit.csv](../outputs/hydrograph_residual_additions_20261019/response_cohort_audit.csv) and [frozen_prediction_coverage.csv](../outputs/hydrograph_residual_additions_20261019/frozen_prediction_coverage.csv).

The completed experiment table and interpretation evidence are recorded at the end of this notebook. No fresh model-selection results should overwrite the frozen baseline.


In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
ROOT=Path.cwd()
while not (ROOT/'RAPID').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Run inside river-hierarchy')
    ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'RAPID/src'))
OUT=ROOT/'RAPID/outputs/hydrograph_frozen_residuals_20261019'
d=pd.read_parquet(OUT/'diagnostic_rows.parquet')
a=pd.read_csv(OUT/'error_attribution.csv',dtype={'group':str})
s=pd.read_csv(OUT/'candidate_screen.csv')
short=pd.read_csv(OUT/'shortlist.csv')
print('Largest network contributions to squared error:')
display(a[a.grouping=='example'].nlargest(5,'sse_share_pct'))
print('Largest network contributions to MAPE:')
display(a[a.grouping=='example'].nlargest(5,'mape_points'))

Largest network contributions to squared error:
Largest network contributions to MAPE:


,grouping,group,n,sse_share_pct,mape_points,relative_points,bias_minutes,mae_minutes
7,example,37,395,47.932363,6.018761,3.749559,0.884428,2.866721
10,example,46,223,9.828952,4.036374,4.861212,-0.612881,4.671720
2,example,05,51,7.110555,2.780447,3.317756,4.721810,4.931833
13,example,55,212,6.686677,3.545044,3.593722,3.935304,4.012310
8,example,41,181,6.113792,4.240188,4.380346,-0.903029,3.474328


,grouping,group,n,sse_share_pct,mape_points,relative_points,bias_minutes,mae_minutes
11,example,48,72,1.778857,10.426039,9.574444,-2.285408,2.333618
12,example,54,81,3.142559,6.590116,7.696796,-1.834378,2.571917
6,example,22,408,4.710544,6.158504,3.679775,-0.869052,1.728295
7,example,37,395,47.932363,6.018761,3.749559,0.884428,2.866721
5,example,21,30,2.310090,5.760316,5.760316,-3.324308,3.342543


## Five focused cases

37 is mainly an extreme extrapolation case. 55 / CB001_bubble_38 is another substantial underpredicted advance. Networks 41 and 46 contain large centroid delays despite peak advances, so a single translated hydrograph is not an adequate explanation. Network 48 supplies the largest total MAPE contribution; CB008 is one of its small-shift examples.

A positive signed residual means the prediction is too late (or its advance is too small); a negative residual means it is too early. MAPE contributions sum to the equal-network score outside ±1 min. Squared-error shares sum to 100%; they are not the change in R² from deleting each row.

In [4]:
CASES=[('37','CB001_bubble_3'),('41','S130_unit_126'),('55','CB001_bubble_38'),('46','S142_unit_37'),('48','CB008_bubble_13')]
review=pd.DataFrame([d[(d.example==ex)&(d.state==state)].iloc[0] for ex,state in CASES])
review['peak_minutes']=review.peak_shift_hours*60
display(review[['example','state','aliases','observed_minutes','predicted_minutes','error_minutes','peak_minutes','sse_share_pct','mape_points']])
px.scatter(d,x='observed_minutes',y='error_minutes',color='example',symbol='edit_type',hover_data=['state','aliases','relative_error_pct','sse_share_pct'],labels={'observed_minutes':'Modelled centroid shift (min)','error_minutes':'Prediction minus response (min)'}).show()

,example,state,aliases,observed_minutes,predicted_minutes,error_minutes,peak_minutes,sse_share_pct,mape_points
892,37,CB001_bubble_3,CB001_bubble_3,-447.688034,-82.889367,364.798667,-525.0,44.288436,0.035505
1429,41,S130_unit_126,S130_unit_126,17.678462,-37.626124,-55.304585,-68.0,2.221389,0.139038
1937,55,CB001_bubble_38,CB001_bubble_38,-97.872545,-39.200948,58.671598,-102.0,2.134523,0.019784
1732,46,S142_unit_37,S142_unit_37 | island_46_037,49.597402,0.914751,-48.682651,-25.0,1.397090,0.031766
1790,48,CB008_bubble_13,CB008_bubble_13,-1.487775,-6.728517,-5.240742,-2.0,0.050146,1.021025


## Candidate evidence and redundancy

The allowlist includes outer-domain parent descriptors, ratios and differences, plus selected-unit and network context descriptors. Routed celerity, depth, discharge and response-shape measurements are **not** candidate predictors.

We require ≥70% equal-network coverage. Spearman correlations are calculated separately within networks; a ranking requires at least eight networks with enough variation. The score `priority` is only a descriptive sorting aid, not a statistical significance test. Partial rank correlations control for the seven baseline features and collapse type. A further response-magnitude-adjusted diagnostic checks whether absolute-error associations simply follow shift size; that response value is never an input predictor.

Parent entropy and expected bifurcations have pooled Spearman correlation **0.996**: use one representative. Changes in cycle rank and bifurcation count correlate **0.963**: do not add both initially. Area-elongation ratios also rank reasonably for signed errors, but combine existing mean-length and width ratios with total-length ratio: R_elongation = R_muL²/(R_W R_Ltot). Prefer testing the underlying new information rather than multiple derived copies.

An absolute-error association is not a signed-error correction. Associations with percentage error and signed residuals are mostly weak (roughly 0.1–0.15), so the shortlist is tentative.

In [6]:
display(short)
cols=['feature','target','networks','mean_within_network_rho','direction_consistency','partial_rank_rho','partial_controlling_response_magnitude','strongest_baseline_correlation']
display(s[(s.view=='all')&s.feature.isin(short.feature)][cols])
# Choose one feature and one residual definition, avoiding a long stack of plots.
FEATURE='B_delta_width_sd'
ERROR='error_minutes'  # alternatively absolute_error_minutes or relative_error_pct
px.scatter(d,x=FEATURE,y=ERROR,color='example',symbol='edit_type',hover_data=['state','aliases','observed_minutes'],title='Exploratory association; no causal or predictive-gain claim').show()

,feature,symbol,reason,role
0,B_delta_width_sd,Δσ_W,Most consistent candidate among relative-error...,width heterogeneity; candidate prediction addi...
1,B_parent_route_entropy,H_p^D,Parent complexity has signed residual associat...,absolute complexity beyond entropy ratio
2,B_delta_total_length,ΔL_tot,Absolute-error rho −0.52 in all 15 networks; w...,extent of edit; error-risk descriptor first
3,B_delta_cycle_rank,Δβ,Absolute-error rho −0.43 with consistent direc...,complexity removed; error-risk descriptor first


,feature,target,networks,mean_within_network_rho,direction_consistency,partial_rank_rho,partial_controlling_response_magnitude,strongest_baseline_correlation
6,B_delta_total_length,error_minutes,15.0,-0.056280,0.533333,0.010230,0.098337,0.555446
7,B_delta_total_length,absolute_error_minutes,15.0,-0.524923,1.000000,-0.236175,-0.096264,0.555446
8,B_delta_total_length,relative_error_pct,15.0,-0.021875,0.600000,-0.013566,-0.018981,0.555446
80,B_delta_width_sd,error_minutes,15.0,0.103801,0.866667,0.097582,0.069502,0.828188
81,B_delta_width_sd,absolute_error_minutes,15.0,-0.010587,0.666667,0.124316,0.069968,0.828188
82,B_delta_width_sd,relative_error_pct,15.0,0.131297,0.733333,0.082312,0.084869,0.828188
173,B_delta_cycle_rank,error_minutes,14.0,-0.065170,0.571429,0.004212,0.015357,0.318476
174,B_delta_cycle_rank,absolute_error_minutes,14.0,-0.432525,1.000000,-0.134486,-0.121839,0.318476
175,B_delta_cycle_rank,relative_error_pct,14.0,-0.011449,0.357143,-0.067466,-0.068100,0.318476
200,B_parent_route_entropy,error_minutes,15.0,0.131672,0.733333,0.135937,0.144876,0.857231


### A concrete complexity hypothesis

For full compound collapses the child route entropy is zero, so R_H is zero regardless of parent complexity. Network 55 / CB001_bubble_38 has parent entropy **6.97** and a **97.9-minute advance**. Its nearest comparator under the baseline's seven training-standardized predictors, 41 / CB009_bubble_28, has parent entropy **1.94** and a **32.9-minute advance**. This motivates testing parent entropy alongside the ratio, but does not prove it explains the difference. The nearest neighbors are chosen without response information, from other networks only; their distances are descriptive, not a formal applicability-domain test.

In [8]:
neighbors=pd.read_csv(OUT/'nearest_baseline_geometry.csv',dtype={'example':str,'query_example':str})
display(neighbors[['query_example','query_state','example','state','distance','observed_minutes','B_parent_route_entropy','B_delta_cycle_rank','B_delta_width_sd']])

,query_example,query_state,example,state,distance,observed_minutes,B_parent_route_entropy,B_delta_cycle_rank,B_delta_width_sd
0,41,S130_unit_126,55,S159_unit_148,0.895560,-27.448935,1.801725,-2.0,70.605269
1,41,S130_unit_126,46,S105_unit_156,0.976586,-3.293772,1.019980,-1.0,35.815325
2,41,S130_unit_126,55,S111_unit_114,0.995646,-10.738695,2.144124,-3.0,43.481126
3,46,S142_unit_37,55,island_55_042,0.466115,-5.254775,4.738597,-1.0,34.197191
4,46,S142_unit_37,55,S150_unit_141,0.502678,-5.116675,3.856490,-1.0,15.941722
5,46,S142_unit_37,22,island_22_192,0.517494,-0.620831,4.880935,-1.0,35.174618
6,55,CB001_bubble_38,41,CB009_bubble_28,0.389546,-32.851294,1.938933,-5.0,-349.312454
7,55,CB001_bubble_38,41,CB004_bubble_25,0.457833,-47.453055,2.525669,-11.0,-338.981728
8,55,CB001_bubble_38,45,CB002_bubble_22,0.689265,-46.799327,2.239790,-6.0,-197.233570


## Hydrograph review

Change CASE to any of the five listed pairs. Curves use the hash-verified series and shared 0.5% window that generated the response tables. Both centroid and sampled peak differences are recomputed and checked. Curves are not peak-aligned. Dotted vertical lines are full excess-volume centroids.

The tabulated duration, volume and asymmetry changes below are supporting diagnostics only. They must not be supplied as geometric predictors.

In [10]:
from rapid_tools.hydrograph_response import load,pulse,distribution
import plotly.graph_objects as go
CASE=('41','S130_unit_126')
states,responses=load('0.5%')
ex,state=CASE
r=d[(d.example==ex)&(d.state==state)].iloc[0]
display(r[['centroid_shift_hours','peak_shift_hours','duration80_change_pct','volume_change_pct','asymmetry_change','peak_excess_change_pct']].to_frame('response'))
fig=go.Figure();centroids=[];peaks=[]
for label,key,color in [('Parent','S000_base','#2563eb'),('Child',state,'#dc2626')]:
    sr=states[(states.example==ex)&(states.state==key)].iloc[0]
    t,q,_=pulse(sr)
    mu=distribution(t,q)['centroid_seconds'];centroids.append(mu);peaks.append(t[q.argmax()])
    fig.add_scatter(x=pd.to_datetime(t,unit='s',utc=True),y=q+sr.baseline_cms,name=label,line=dict(color=color))
    fig.add_vline(x=mu*1000,line_color=color,line_dash='dot')
np.testing.assert_allclose((centroids[1]-centroids[0])/3600,r.centroid_shift_hours,atol=1e-8)
np.testing.assert_allclose((peaks[1]-peaks[0])/3600,r.peak_shift_hours,atol=1e-8)
fig.update_layout(title=f'{ex} / {state}: finalized shared window',xaxis_title='UTC',yaxis_title='Discharge (m³/s)',template='plotly_white')
fig.show()

,response
centroid_shift_hours,0.294641
peak_shift_hours,-1.133333
duration80_change_pct,0.757468
volume_change_pct,-0.065991
asymmetry_change,0.006441
peak_excess_change_pct,-0.864979


## Recorded investigation decisions — 19 October 2026

**Status:** retain the frozen seven-input forest. The additional tests below do not replace it. All are complete; no statistical companion model or fresh SHAP analysis of the final frozen forest has been run in this update.

**Data and target:** 2,170 comparisons across **15 examples: 02, 03, 05, 13, 15, 21, 22, 37, 41, 45, 46, 48, 54, 55, 56**. Predict full excess-volume centroid shift, child minus parent, in hours. The shared 0.5% analysis-window response conventions remain unchanged. Peak-timing scores transfer these centroid predictions; they are not a separately fitted peak model. Example 54 retains its defined response despite crest switching and its elevated ending equilibrium.

**Baseline:** seven inputs in the existing outer comparison domain: logged child/parent ratios of mean travelled length, route-length SD, mean width, width CV and route entropy; parent mean travelled length divided by 1 km; and parent mean split dominance. Resistance and exposure are excluded. Forest: 500 trees, absolute_error, max_depth=16, min_samples_leaf=2, min_samples_split=20, max_features=1.0, bootstrap=True, max_samples=None, ccp_alpha=0.001, random_state=20261014. Training weights give examples equal total weight; preprocessing and target standardization are learned within training folds.

**Validation:** each of 15 networks held out in turn; identical settings and rows for every new test. Relative loss is equally weighted across examples with denominator max(abs(observed shift), 1 minute). Settings were selected previously using these examples: these are exploratory fixed-setting comparisons, not an independent test or fresh nested tuning. Excluding 54 or the extreme 37 case from the tables below means evaluation-only exclusion, not a new fit excluding them from all training folds. The earlier nested-tuning centroid R² ≈0.6044 should remain distinguished from the fixed-settings baseline R² ≈0.6145.

### Added-predictor experiments

All additions use the existing outer domain D. Differences are signed child minus parent; no domain changes or new routed hydraulic inputs were introduced.

- `delta_entropy`: replace ln(1+Hc/Hp) with Hc−Hp; seven inputs.
- `ratio_plus_parent_entropy`: retain the ratio and add Hp; eight inputs.
- `width_sd`: add ΔσW; eight inputs.
- `total_length`: add change in summed unique channel length; eight inputs.
- `cycle_rank`: add change in cycle rank; eight inputs.
- `all_three`: add ΔσW, ΔLtot and Δcycle rank; ten inputs. Pairwise combinations were not tested.

| Experiment | R² | MAE (minutes) | Relative error (%) |
| --- | --- | --- | --- |
| seven_inputs | 0.6145 | 2.8224 | 63.7372 |
| delta_entropy | 0.5904 | 2.8224 | 65.9614 |
| ratio_plus_parent_entropy | 0.6115 | 2.8025 | 63.6855 |
| width_sd | 0.6141 | 2.7951 | 64.4339 |
| total_length | 0.5941 | 2.9268 | 67.412 |
| cycle_rank | 0.611 | 2.8352 | 64.8365 |
| all_three | 0.5927 | 2.9223 | 67.2917 |

No addition improved relative error enough to justify replacing the baseline. Parent entropy changes 63.7372% to 63.6855%, a negligible exploratory gain accompanied by lower R². Width SD improves MAE slightly but worsens overall relative error. It does improve relative error in the zero-child-entropy subgroup, 75.05% to 70.49%, so retain it as a possible subgroup diagnostic, not an accepted predictor. Results without the extreme 37 case do not reverse the overall recommendation. These results do not prove the variables have no physical relevance or cannot help under another fitted model.

Full response/subgroup comparisons: [investigation_comparison.csv](../outputs/hydrograph_residual_additions_20261019/investigation_comparison.csv). Reproducible scripts: `delta_entropy_check.py`, `parent_entropy_check.py`, `residual_additions_check.py` under `RAPID/investigations/hydrograph_predictors_20261014/`. Each experiment has source hashes, predictions, fold checkpoints, scores and a separately serialized model. Frozen artifact: `RAPID/outputs/hydrograph_no_exposure_20261019/final_model.joblib`.

### Why entropy was investigated

436/2,170 children (20.1%) have zero outer-domain route entropy: all 258 simple units, all 177 full compounds, one compound member, no islands. Their parent entropy spans 0.171–44.186. Zero is physically meaningful (no remaining splits), but the ratio maps all to zero. Spearman(ΔH, Hc/Hp)=0.9039 overall and 0.8731 among nonzero children; median within-example correlation=0.8667. Zero-child cases account for 58/100 largest absolute errors and 76.6% of weighted squared error, or 58.0% after evaluating without 37/CB001_bubble_3. This motivated the tests; association did not guarantee predictive improvement.

The 92-representation residual screen remains exploratory. Parent entropy and expected split count were nearly redundant (Spearman≈0.996); changes in cycle rank and bifurcation count were also highly related (≈0.963). Width SD had weak recurring signed/relative-error associations. Total length and cycle-rank changes mainly tracked absolute-error risk. No causal interpretation or formal significance is established by these screens.

### Existing SHAP and permutation-importance evidence — earlier model version

**Provenance warning:** these numbers describe the earlier relative-sweep forests, reconstructed per held-out network, including the constant exposure input. They are NOT recomputed SHAP values for the final tuned seven-input frozen forest. Source: `hydrograph_forest_interpretation_20261019`, generated by `forest_interpretation.py`; see its design and interpretation manifests. Existing analysis verified held-out predictions and SHAP additivity, with 20 within-held-network permutation repeats and equal-example aggregation.

| Predictor | Mean absolute SHAP (minutes) | Permutation relative-loss increase (pp) |
| --- | --- | --- |
| R_W | 6.97 | 182.881 |
| muL_Dp / 1 km | 5.297 | 75.1 |
| R_muL | 0.673 | 24.732 |
| R_H | 0.439 | 5.29 |
| R_sigmaL | 0.122 | 2.38 |
| d_Dp | 0.107 | 0.408 |
| R_CVW | 0.097 | 1.082 |
| r_in | 0.0 | 0.0 |

Width ratio and parent-domain size dominate both summaries; mean travelled-length ratio follows. Exposure contributes zero. SHAP magnitude is allocation of model prediction, not an accuracy gain. Permutation measures held-out loss sensitivity; correlated inputs and implausible shuffled combinations affect it, so individual and grouped results must be considered together. Neither quantity is a p-value, an independent physical effect, or evidence that a variable causes the response. Existing SHAP dependence and interaction plots are in [hydrograph_forest_interpretation.ipynb](hydrograph_forest_interpretation.ipynb).

### Co-author question: an interpretable statistical companion

A reasonable first companion is a **multiple linear regression with a network random intercept**, using the same seven inputs:

$$y_{ig}=\alpha+b_g+\sum_j\beta_j x_{ij}+\varepsilon_{ig}.$$

Coefficients describe conditional associations on the specified transformed-input scale, under the model assumptions; they are not automatically causal effects. Network effects address shared example context but do not guarantee all dependence among shared-parent comparisons is removed. Check residual structure and collinearity, avoid treating 2,170 comparisons as independent network replications, and assess coefficient stability by leaving entire networks out. There are only 15 network groups. For a held-out unseen network use the population prediction (random effect zero), never estimate its random intercept from its response.

If linear effects are inadequate, a restrained **generalized additive model (GAM)** is the natural next companion:

$$y_{ig}=\alpha+b_g+\sum_j f_j(x_{ij})+\varepsilon_{ig}.$$

Smooth partial-effect curves show direction, magnitude and nonlinearity with uncertainty, rather than forcing each variable to have one constant slope. Keep complexity low; any selected interaction needs grouped validation. Compare both against the forest using the same response, domain, held-network folds and relative-error/R²/MAE reporting. Distinguish a model's fitting loss from the relative-error criterion used to assess/tune it. Earlier ridge/elastic-net/spline prediction fits are not automatically a completed inferential analysis with suitable dependence-aware uncertainty.

**Complementary interpretation plan (not yet executed):**
1. Refresh held-network permutation importance and SHAP for the actual frozen seven-input model before publication.
2. Show pooled and within-network predictor–response Spearman correlations, predictor redundancy, and cross-network sign stability. Correlations are marginal associations, not adjusted effects; disagreement can reveal confounding or nonlinearity.
3. Show signed SHAP dependence and preferably accumulated-local-effect curves in well-supported predictor ranges, so magnitude rankings are accompanied by direction and shape. Inspect correlated predictors jointly.
4. Fit a small statistical companion if explicit coefficients/effect curves and uncertainty are needed; assess network-level stability and sensitivity to 54 and the extreme 37 comparison. The same data have informed many choices, so treat resulting inference as exploratory.

This combination supports physical interpretation but should be called predictive importance and conditional association, not causal or statistical significance inferred from SHAP alone.

Sources: [mgcv GAM documentation](https://stat.ethz.ch/R-manual/R-devel/library/mgcv/html/gam.html) describes smooth terms, random effects and uncertainty; [SHAP caution on causal interpretation](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/Be%20careful%20when%20interpreting%20predictive%20models%20in%20search%20of%20causal%20insights.html).


## Planned statistical interpretation and companion-model analysis

**Scope:** explain associations with the simulated full-centroid response, while keeping the seven-variable forest as the prediction reference. Distinguish predictive usefulness, conditional association, and causal physical effects. This is an exploratory analysis after substantial model development, not a new confirmatory significance study. Do not use p<0.05 as an automatic variable-selection rule.

1. **Lock cohort, inputs and estimand.** Use the current 2,170 comparisons, seven transforms, outer domains and signed response. Preserve equal-example evaluation. Audit shared-parent identities and the two missing partial-collapse records; document whole-network states separately. Before predicting single-centerline states, verify seven-feature availability and domain equivalence, and report them separately rather than silently expanding the baseline. Record data/model hashes and existing scores.
2. **Refresh interpretation for the actual frozen model.** Reconstruct each of the 15 fixed-settings held-network forests; verify predictions against saved seven-input predictions. Compute held-network individual and grouped permutation importance (20 repeats initially), SHAP magnitudes and signed dependence plots, plus well-supported accumulated-local-effect curves. Aggregate equally over examples; report per-network stability. Repeat diagnostic summaries without 54 and without the extreme 37 comparison, explicitly distinguishing scoring-only exclusions from refits. Historical SHAP remains separately labeled.
3. **Describe observed associations and redundancy.** Produce predictor–predictor correlations, pooled and within-network predictor–response Spearman correlations, support plots and sign stability. Do not interpret pooled correlation as an adjusted effect. Compare these with signed SHAP/effect curves; disagreement is evidence to investigate, not a reason to cherry-pick a preferred method. Constant within-network variables have undefined within-network correlation. Avoid naive independent-row correlation p-values.
4. **Fit a compact linear statistical companion.** Start with the same seven inputs in a linear mixed model with a network random intercept, y_ig=alpha+b_g+sum(beta_j*x_ij)+epsilon_ig. Fit using Gaussian likelihood/REML rather than claiming it minimizes the forest's relative-error objective. Report the fitting objective and row/group weighting explicitly; a random intercept is not equal-network weighting. Include an equal-example-weighted linear regression sensitivity to separate the influence of weighting from model form. Standardize within training folds for predictive comparisons and document reference scales for coefficient interpretation. Check collinearity, residual curvature/heteroscedasticity, influential cases, random-effect convergence and shared-parent dependence. Network random effects can correlate with predictors: compare within-network centered effects or network-fixed-effect sensitivity if between-network and within-network associations differ. Do not add a large random-slope structure with only 15 groups.
5. **Quantify uncertainty and practical effect size.** Prioritize coefficient estimates, meaningful predictor contrasts, 95% intervals and leave-one-network-out sign/range stability. Use resampling of entire networks for exploratory uncertainty, keeping all within-network comparisons together (fresh group IDs for duplicated bootstrap networks); report failure rates and the limitation of only 15 groups. Predeclare the seven coefficient tests if p-values are requested; use a validated small-cluster procedure appropriate to the fitted model, not ordinary row-wise asymptotic tests, and report raw plus Holm-adjusted values. Bootstrap intervals and significance do not remove post-selection bias from earlier exploration. Separate uncertainty about simulated-network associations from uncertainty about real-world hydrology.
6. **Use a small GAM only if linear effects are inadequate.** Keep the same predictors; low-rank penalized smooths (initial basis dimension 4), network random intercept, no default interaction sweep. Inspect effective degrees of freedom and concurvity; select smoothing using training data only. Smooth-effect plots with uncertainty provide direction and shape, not a single coefficient. Any later width–size interaction is a separately declared extension. Approximate GAM p-values have smoothing-uncertainty limitations; do not equate smooth significance with causal importance.
7. **Compare predictive and explanatory value.** Use identical leave-one-network-out folds and equal-example floored relative error, R² and MAE. A new held-out network receives the population prediction, with network random effect zero; no fitting its intercept from held-out outcomes. Report both all-case and subgroup errors, and perform explicit refit sensitivities excluding 54 and the extreme 37 case if assessing coefficient/model stability. Peak shift remains a transfer diagnostic unless a separate model is explicitly fitted. Report where the simpler model loses predictive skill and whether its effect directions are stable. No untouched test is created by reusing these folds.

**Deliverables:** a current-model importance notebook; a coefficient/effect-curve and uncertainty table; a matched performance comparison; a short physical interpretation with coverage and extrapolation limitations. First implementation stage should complete steps 1–3 plus the linear companion; GAM and larger uncertainty runs follow only if justified. This plan does not authorize claiming any new statistical findings before running it.

References: [MixedLM group/random-intercept model](https://www.statsmodels.org/stable/_modules/statsmodels/regression/mixed_linear_model.html), [GAM model specification](https://stat.ethz.ch/R-manual/R-devel/library/mgcv/html/gam.html), [limitations of approximate GAM significance tests](https://stat.ethz.ch/R-manual/R-devel/library/mgcv/html/summary.gam.html).


## Whole-network collapse side evaluation — completed 19 October 2026

The seven geometric predictors **are defined** for all 15 single-centerline comparisons. Existing scope extraction uses B=N (the whole network), with closed-domain status validated. All inputs are finite. The saved scope values match the frozen training inputs for all 2,170 existing comparisons; full-collapse centroid/peak responses match the finalized response table. No new geometric definitions were invented.

Child route entropy and route-length SD are zero; width CV is zero to floating-point precision because these single-centerline models have uniform width. These are valid quantities. Parent mean travelled length spans 61.74–810.36 km. In 12/15 held-network folds this exceeds the training-domain maximum (typically 102.57 km); example 37 also exceeds the held-training mean-width-ratio maximum. Univariate range coverage does not guarantee joint-distribution coverage.

**Protocol:** reconstruct the frozen seven-input forest separately for each held network, trained only on the other networks' partial collapses; verify its predictions reproduce the saved baseline predictions before evaluating that held network's whole-network collapse. All 15 verification checks passed. No whole-network response was used for training or feature construction. This is a separate transfer diagnostic, not an augmented training run or an independent external test.

| Target evaluated | R² | MAE | Relative error (1-minute floor) |
|---|---:|---:|---:|
| Full centroid shift | −0.575 | 617.96 min | 82.63% |
| Peak shift (centroid prediction transferred) | −1.125 | 704.29 min | 82.21% |

All 15 are advances and all are underestimated in magnitude. Example 46: actual centroid shift −3473.92 min (−57.90 h), predicted −64.72 min (−1.08 h). Example 55: actual −1114.41 min, predicted −61.85 min. This establishes poor transfer by the current model; it does not establish that geometry is inherently unsuitable. Scale extrapolation is a clear concern, alongside the changed edit scope and potential missing information. The geometrically valid zeros are not themselves a failure.

**Decision:** include these states in reporting as a separate side evaluation, retain the frozen baseline unchanged, and do not pool these scores with the 2,170 partial-collapse scores. If augmenting training is later pursued, use the 14 training networks' whole-collapse cases inside each held-network fold, hold out every comparison from the test network, explicitly choose whole-collapse weighting, and report partial versus whole-collapse results separately. Adding 15 training observations does not guarantee reliable extrapolation to the largest networks.

Artifacts: [predictions.csv](../outputs/hydrograph_whole_network_check_20261019/predictions.csv), [predictor_support.csv](../outputs/hydrograph_whole_network_check_20261019/predictor_support.csv), [scores.csv](../outputs/hydrograph_whole_network_check_20261019/scores.csv). Reproducible script: `RAPID/investigations/hydrograph_predictors_20261014/whole_network_check.py`. Source hashes, input table and resumable fold checkpoints are saved alongside results. No augmented model has been fitted.


## Planned adjacent-pair and segmented whole-network experiment

The coordinated, implementation-grounded plan is in [hydrograph_closed_unit_pairing_plan.ipynb](hydrograph_closed_unit_pairing_plan.ipynb). It restricts selections to adjacent closed simple/full outer-compound units, preserves connectors and boundary nodes, and proposes a six-pair/three-network pilot. A separately versioned all-block single-path variant reuses the same operation. Legacy reference geometry/calibration and frozen model results remain intact. Implementation and simulations are pending.